# 05_kde_infer_aggregate.ipynb
Inference + KDE-style aggregation for unseen locations under B.


In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd, tensorflow as tf
BUILD_DIR=Path('dataset_build_hybrid'); MODEL_DIR=Path('transfer_B_runs'); OUT_DIR=Path('infer_runs_unseen_locations'); OUT_DIR.mkdir(exist_ok=True)
def load_npz(name): o=np.load(BUILD_DIR/f'{name}.npz',allow_pickle=True); return {k:o[k] for k in o.files}
target_eval_unseen_locations=load_npz('target_eval_unseen_locations'); B_finetune=load_npz('B_finetune')
assert set(np.unique(target_eval_unseen_locations['receiver_domain'].astype(str)))=={'B'}
model=tf.keras.models.load_model(MODEL_DIR/'hybrid_transfer_B_best.keras',compile=False)


In [ ]:
rows=[]
for i in range(len(target_eval_unseen_locations['amp_path'])):
    amp=np.load(str(target_eval_unseen_locations['amp_path'][i])).astype('float32'); pha=np.load(str(target_eval_unseen_locations['pha_path'][i])).astype('float32') if Path(str(target_eval_unseen_locations['pha_path'][i])).exists() else np.zeros_like(amp)
    if amp.ndim==2: amp=amp[...,None]
    if pha.ndim==2: pha=pha[...,None]
    prob=model.predict({'amp_in':amp[None,...],'pha_in':pha[None,...]},verbose=0)[0]
    rows.append({'sample_id':str(target_eval_unseen_locations['sample_id'][i]),'anchor_label':str(target_eval_unseen_locations['anchor_label'][i]),'session_id':str(target_eval_unseen_locations['session_id'][i]),'receiver_domain':'B','pred_class':int(np.argmax(prob)),'pred_confidence':float(np.max(prob))})
pred_df=pd.DataFrame(rows)


In [ ]:
agg_df=pred_df.groupby(['anchor_label','session_id']).apply(lambda x: pd.Series({'n_windows':len(x),'avg_confidence':float(x.pred_confidence.mean()),'mode_pred_class':int(x.pred_class.mode().iloc[0])})).reset_index()
pred_df.to_csv(OUT_DIR/'target_eval_unseen_locations_window_predictions.csv',index=False)
agg_df.to_csv(OUT_DIR/'target_eval_unseen_locations_aggregated.csv',index=False)
report={'evaluation_target':'prediction_on_unseen_locations_under_B','used_split':'target_eval_unseen_locations','adaptation_split_separate':'B_finetune','single_receiver_only':True,'no_AB_joint_aggregation':True,'n_eval_windows':int(len(pred_df))}
(OUT_DIR/'kde_infer_summary.json').write_text(json.dumps(report,indent=2)); print(json.dumps(report,indent=2))
